**`03_prepare_administrative_units`**

Prepare initial layer of administrative units with geometries and `AdminId` identifiers.

In [1]:
%load_ext autoreload
%autoreload 2

# Administrative units identifiers
``openplaces`` organizes its data by ``admin_id`` (data class: ``AdminId``).

``admin_id`` is a geographical administrative index with hierarchical ``.levels`` of any depth.

- **0** - countries
- **1** - states/departments/...
- **2** - counties/municipalities/...
- **3** - subdivisions/towns/...
- **4** - neighborhoods/...
- **5** - ...

The initial built is derived from ISO and the Global Administrative Database (GADM).

In [2]:
from openplaces.api import get_admin0, get_admin1, get_admin2
from openplaces.core.schema import AdminId
from openplaces.io.admin import get_admin0_iso, get_admin1_iso
from openplaces.io.ingest import ingest_recipe
from openplaces.recipe import get_recipe
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [3]:
# Overwrite existing outputs?
REDO = False

## Start timer

In [4]:
timer = get_timer('import_admin', verbose=True)

# `admin0`: countries / territories
The highest level of the administrative hierarchy.
## ISO countries
Top-level administrative identifiers, gap-filled to match GADM, ships with `openplaces`

In [5]:
get_admin0_iso()

,name,admin0_id_a3,region,sub-region,intermediate-region
admin0_id,,,,,
AD,Andorra,AND,Europe,Southern Europe,
AE,United Arab Emirates,ARE,Asia,Western Asia,
AF,Afghanistan,AFG,Asia,Southern Asia,
AG,Antigua and Barbuda,ATG,Americas,Latin America and the Caribbean,Caribbean
AI,Anguilla,AIA,Americas,Latin America and the Caribbean,Caribbean
...,...,...,...,...,...
Z9,India - Z9,Z09,Asia,Southern Asia,
ZA,South Africa,ZAF,Africa,Sub-Saharan Africa,Southern Africa
ZM,Zambia,ZMB,Africa,Sub-Saharan Africa,Eastern Africa


## GADM level 0
Example of how to download and ingest data from the Internet using a `recipe`:

In [6]:
recipe = get_recipe(AdminId(), 'admin', source='admin0-gadm-4~1')
pretty_print(recipe)

admin_id:
  levels: []
entity:
  entity_type:
    entity_type: admin
  source:
    source_id: gadm
    url: https://geodata.ucdavis.edu/gadm/gadm4.1/gadm_410-levels.zip
    doi: null
  version: 4~1
compressed_file_name: gadm_410-levels.zip
uncompressed_file_name: gadm_410-levels.gpkg
layer: ADM_0
columns:
  name: COUNTRY
  admin0_id_a3: GID_0
index_function: openplaces.io.admin.admin0_id_index_from_admin0_id_a3
cache_filename: admin0
compute_poi: true
simplify_coverage_tolerance: 0.1


In [7]:
ingest_recipe(recipe, timer=timer, redo=REDO)

### Read result

In [8]:
admin0 = get_admin0()
admin0.sample(5).T

admin0_id,BB,MS,CI,IM,LK
name,Barbados,Montserrat,Côte d'Ivoire,Isle of Man,Sri Lanka
admin0_id_a3,BRB,MSR,CIV,IMN,LKA
lat,13.172262,16.739452,7.625571,54.22758,7.621672
long,-59.55639,-62.189685,-5.555422,-4.539159,80.697868
ha,43463.964109,10064.547337,32154324.510363,57967.292884,6583740.275246
x_poi,-59.549305,-62.190157,-5.547095,-4.589027,80.846017
y_poi,13.143697,16.736348,7.442149,54.174649,7.209751
r_poi,8485.037517,4544.546242,255422.409393,13530.27802,106364.960033


# ``admin1``: states / departments

## ISO states

In [9]:
admin1_iso = get_admin1_iso()
admin1_iso

,name,admin0_id,admin0_name
admin1_id_iso3166,,,
AD-02,Canillo,AD,Andorra
AD-03,Encamp,AD,Andorra
AD-04,La Massana,AD,Andorra
AD-05,Ordino,AD,Andorra
AD-06,Sant Julia de Loria,AD,Andorra
...,...,...,...
ZW-MI,Midlands,ZW,Zimbabwe
ZW-MN,Matabeleland North,ZW,Zimbabwe
ZW-MS,Matabeleland South,ZW,Zimbabwe


## GADM level 1

In [10]:
recipe = get_recipe(AdminId(), 'admin', source='admin1-gadm-4~1')
pretty_print(recipe)

admin_id:
  levels: []
entity:
  entity_type:
    entity_type: admin
  source:
    source_id: gadm
    url: https://geodata.ucdavis.edu/gadm/gadm4.1/gadm_410-levels.zip
    doi: null
  version: 4~1
compressed_file_name: gadm_410-levels.zip
uncompressed_file_name: gadm_410-levels.gpkg
layer: ADM_1
columns:
  name: NAME_1
  type: ENGTYPE_1
  admin0_name: COUNTRY
  name_original: NL_NAME_1
  name_alternatives: VARNAME_1
  type_original: TYPE_1
  admin1_id_gadm: GID_1
  admin1_id_hasc: HASC_1
  admin1_id_iso: ISO_1
  admin1_id_original: CC_1
  admin0_id_a3: GID_0
filter_query: type != 'Water body'
null_value_strings:
  - ?
index_function: openplaces.io.admin.admin1_id_index_from_admin1_gadm
cache_filename: admin1
compute_poi: true
simplify_coverage_tolerance: 0.1


In [11]:
ingest_recipe(recipe, timer=timer, redo=REDO)

In [12]:
admin1 = get_admin1()
admin1.sample(5).T

admin1_id,TH-CR,SC-GM,DJ-AR,CH-LU,CN-GD
name,Chiang Rai,Grand' Anse,Arta,Lucerne,Guangdong
type,Province,District,Region,Canton,Province
admin0_name,Thailand,Seychelles,Djibouti,Switzerland,China
name_original,จังหวัดเชียงราย,NA,إقليم عرتا,NA,廣東|广东
name_alternatives,NA,NA,NA,Lucerna|Luzern,Guǎngdōng
type_original,Changwat,District,NA,Kanton,Shěng
admin1_id_gadm,THA.11_1,SYC.14_1,DJI.6_1,CHE.12_1,CHN.6_1
admin1_id_hasc,TH.CR,SC.GM,DJ.AR,CH.LU,CN.GD
admin1_id_iso,TH-57,NA,DJ-AR,NA,CN-GD
admin1_id_original,57,NA,NA,NA,NA


# ``admin2``: counties / municipalities

## GADM level 2

In [13]:
recipe = get_recipe(AdminId(), 'admin', source='admin2-gadm-4~1')
pretty_print(recipe)

admin_id:
  levels: []
entity:
  entity_type:
    entity_type: admin
  source:
    source_id: gadm
    url: https://geodata.ucdavis.edu/gadm/gadm4.1/gadm_410-levels.zip
    doi: null
  version: 4~1
compressed_file_name: gadm_410-levels.zip
uncompressed_file_name: gadm_410-levels.gpkg
layer: ADM_2
columns:
  name: NAME_2
  type: ENGTYPE_2
  admin1_name: NAME_1
  admin0_name: COUNTRY
  name_original: NL_NAME_2
  name_alternatives: VARNAME_2
  type_original: TYPE_2
  admin1_type: TYPE_1
  admin1_name_original: NL_NAME_1
  admin2_id_gadm: GID_2
  admin2_id_original: CC_2
  admin2_id_hasc: HASC_2
  admin1_id_gadm: GID_1
  admin0_id_a3: GID_0
filter_query: type not in ('Water body', 'Water Body')
index_function: openplaces.io.admin.admin2_id_index_from_admin2_gadm
cache_filename: admin2
compute_poi: true
simplify_coverage_tolerance: 0.1


In [14]:
ingest_recipe(recipe, timer=timer, redo=REDO)

In [15]:
admin2 = get_admin2()
admin2.sample(5).T

admin2_id,TH-SS-KB,BR-PA-CUR,BR-CE-TDN,MX-QE-AS,PH-SQ-SI
name,Krathum Baen,Curionópolis,Tabuleiro do Norte,Arroyo Seco,Siquijor
type,District,Municipality,Municipality,Municipality,Municipality
admin1_name,Samut Sakhon,Pará,Ceará,Querétaro,Siquijor
admin0_name,Thailand,Brazil,Brazil,México,Philippines
name_original,อำเภอกระทุ่มแบน,NA,NA,NA,NA
name_alternatives,NA,NA,NA,NA,NA
type_original,Amphoe,Município,Município,Município,Bayan|Munisipyo
admin1_name_original,จังหวัดสมุทรสาคร,NA,NA,NA,NA
admin2_id_gadm,THA.58.2_1,BRA.14.42_2,BRA.6.169_2,MEX.22.2_2,PHL.68.6_1
admin2_id_original,7402,1502772,2313104,NA,76106


# Wrap up

In [16]:
timer.summary()
timer.save()

Timer: import_admin
Total: 1.61s

  _final....................................................     1.61s      1.84s cpu (100.0%)
